<a href="https://colab.research.google.com/github/pranchalkumar001/TruePatient/blob/main/CNN%2BLLaVA_1_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch

# Force CPU (no CUDA at all)
os.environ["CUDA_VISIBLE_DEVICES"] = ""
device = torch.device("cpu")

print("Torch version:", torch.__version__)
print("Device:", device)


Torch version: 2.9.0+cpu
Device: cpu


In [ ]:
# Check main dataset folder
!ls /content


sample_data


In [ ]:
# List everything inside /content
!ls /content


sample_data


In [ ]:
!pip install -q opendatasets


In [ ]:
import opendatasets as od

od.download("https://www.kaggle.com/datasets/mehradaria/leukemia")


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: pranchalkumar
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/mehradaria/leukemia


100%|██████████| 110M/110M [00:00<00:00, 2.21GB/s]

In [ ]:
# Check content directory
!ls /content


leukemia  sample_data


In [ ]:
# Inspect the leukemia dataset folder
!ls /content/leukemia


Original  Segmented


In [ ]:
# Check class folders
!ls /content/leukemia/Original


Benign	Early  Pre  Pro


In [ ]:
import os

base_dir = "/content/leukemia_processed"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

classes = ["Benign", "Early", "Pre", "Pro"]

for split in [train_dir, test_dir]:
    for cls in classes:
        os.makedirs(os.path.join(split, cls), exist_ok=True)

print("Train/Test folder structure created successfully")


Train/Test folder structure created successfully


In [ ]:
import os
import shutil
import random

source_dir = "/content/leukemia/Original"
base_dir = "/content/leukemia_processed"

train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

classes = ["Benign", "Early", "Pre", "Pro"]
split_ratio = 0.8

for cls in classes:
    cls_path = os.path.join(source_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    random.shuffle(images)
    split_point = int(len(images) * split_ratio)

    train_imgs = images[:split_point]
    test_imgs = images[split_point:]

    for img in train_imgs:
        shutil.copy(
            os.path.join(cls_path, img),
            os.path.join(train_dir, cls, img)
        )

    for img in test_imgs:
        shutil.copy(
            os.path.join(cls_path, img),
            os.path.join(test_dir, cls, img)
        )

print("Image splitting completed successfully")


Image splitting completed successfully


In [ ]:
# Verify folder structure
!find /content/leukemia_processed -maxdepth 2 -type d


/content/leukemia_processed
/content/leukemia_processed/train
/content/leukemia_processed/train/Benign
/content/leukemia_processed/train/Early
/content/leukemia_processed/train/Pre
/content/leukemia_processed/train/Pro
/content/leukemia_processed/test
/content/leukemia_processed/test/Benign
/content/leukemia_processed/test/Early
/content/leukemia_processed/test/Pre
/content/leukemia_processed/test/Pro


In [ ]:
from torchvision import datasets, transforms

# Simple, safe transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dir = "/content/leukemia_processed/train"
test_dir = "/content/leukemia_processed/test"

train_data = datasets.ImageFolder(train_dir, transform=transform)
test_data = datasets.ImageFolder(test_dir, transform=transform)

print("Train classes:", train_data.classes)
print("Class to index:", train_data.class_to_idx)
print("Number of training images:", len(train_data))
print("Number of test images:", len(test_data))


Train classes: ['Benign', 'Early', 'Pre', 'Pro']
Class to index: {'Benign': 0, 'Early': 1, 'Pre': 2, 'Pro': 3}
Number of training images: 2604
Number of test images: 652


In [ ]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

# Sanity check labels
labels_seen = set()
for _, labels in train_loader:
    labels_seen.update(labels.tolist())

print("Labels seen in training data:", sorted(labels_seen))
print("Expected labels:", list(range(len(train_data.classes))))


Labels seen in training data: [0, 1, 2, 3]
Expected labels: [0, 1, 2, 3]


In [ ]:
import torch.nn as nn
from torchvision import models

# Load pretrained ResNet18
model = models.resnet18(pretrained=True)

# Replace final layer for 4 classes
num_classes = len(train_data.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Move model to CPU
model = model.to(device)

print("Model created successfully")
print("Number of output classes:", num_classes)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 104MB/s]


Model created successfully
Number of output classes: 4


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Loss function and optimizer initialized")


Loss function and optimizer initialized


In [ ]:
# Train for just 1 epoch (safe start)
epochs = 1

model.train()

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch+1} | Loss: {running_loss:.4f} | Accuracy: {acc:.2f}%")


Epoch 1 | Loss: 19.6308 | Accuracy: 96.12%


In [ ]:
import torch
print(torch.cuda.is_available())


True


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda")
print("Using device:", device)


Using device: cuda


In [ ]:
!ls /content


sample_data


In [ ]:
!pip install -q opendatasets


In [ ]:
import opendatasets as od

od.download("https://www.kaggle.com/datasets/mehradaria/leukemia")


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: pranchalkumar
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/mehradaria/leukemia


100%|██████████| 110M/110M [00:00<00:00, 1.01GB/s]

In [ ]:
!ls /content


leukemia  sample_data


In [ ]:
!ls /content/leukemia


Original  Segmented


In [ ]:
!ls /content/leukemia/Original


Benign	Early  Pre  Pro


In [ ]:
import os

base_dir = "/content/leukemia_processed"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

classes = ["Benign", "Early", "Pre", "Pro"]

for split in [train_dir, test_dir]:
    for cls in classes:
        os.makedirs(os.path.join(split, cls), exist_ok=True)

print("Train/Test folder structure created")


Train/Test folder structure created


In [ ]:
import os
import shutil
import random

source_dir = "/content/leukemia/Original"
base_dir = "/content/leukemia_processed"

train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

classes = ["Benign", "Early", "Pre", "Pro"]
split_ratio = 0.8

for cls in classes:
    cls_path = os.path.join(source_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    random.shuffle(images)
    split_point = int(len(images) * split_ratio)

    train_imgs = images[:split_point]
    test_imgs = images[split_point:]

    for img in train_imgs:
        shutil.copy(
            os.path.join(cls_path, img),
            os.path.join(train_dir, cls, img)
        )

    for img in test_imgs:
        shutil.copy(
            os.path.join(cls_path, img),
            os.path.join(test_dir, cls, img)
        )

print("Image split completed successfully")


Image split completed successfully


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Verify folder structure
print("Train folders:", os.listdir("/content/leukemia_processed/train"))
print("Test folders:", os.listdir("/content/leukemia_processed/test"))

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dir = "/content/leukemia_processed/train"
test_dir  = "/content/leukemia_processed/test"

train_data = datasets.ImageFolder(train_dir, transform=transform)
test_data  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)

print("Classes:", train_data.classes)
print("Class to index:", train_data.class_to_idx)
print("Train images:", len(train_data))
print("Test images:", len(test_data))


Train folders: ['Pre', 'Pro', 'Early', 'Benign']
Test folders: ['Pre', 'Pro', 'Early', 'Benign']
Classes: ['Benign', 'Early', 'Pre', 'Pro']
Class to index: {'Benign': 0, 'Early': 1, 'Pre': 2, 'Pro': 3}
Train images: 2604
Test images: 652


In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(train_data.classes))
model = model.to(device)

print("Model loaded on GPU")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 195MB/s]


Model loaded on GPU


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print("Loss function and optimizer ready")


Loss function and optimizer ready


In [ ]:
epochs = 3

model.train()

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {running_loss:.4f} | Accuracy: {acc:.2f}%")


Epoch [1/3] | Loss: 10.1250 | Accuracy: 95.66%
Epoch [2/3] | Loss: 1.0383 | Accuracy: 99.77%
Epoch [3/3] | Loss: 0.4586 | Accuracy: 99.85%


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = 100 * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")


Test Accuracy: 99.54%


In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": train_data.classes
}, "leukemia_resnet18_4class.pth")

print("Model saved as leukemia_resnet18_4class.pth")


Model saved as leukemia_resnet18_4class.pth


In [ ]:
checkpoint = torch.load("leukemia_resnet18_4class.pth", map_location=device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(checkpoint["class_names"]))
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

class_names = checkpoint["class_names"]
print("Model reloaded with classes:", class_names)


Model reloaded with classes: ['Benign', 'Early', 'Pre', 'Pro']


In [ ]:
from PIL import Image
import torch

def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    img = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    return class_names[pred.item()], conf.item() * 100

print("predict_image function is ready")


predict_image function is ready


In [ ]:
!ls /content/leukemia/Original/Pro | head


WBC-Malignant-Pro-001.jpg
WBC-Malignant-Pro-002.jpg
WBC-Malignant-Pro-003.jpg
WBC-Malignant-Pro-004.jpg
WBC-Malignant-Pro-005.jpg
WBC-Malignant-Pro-006.jpg
WBC-Malignant-Pro-007.jpg
WBC-Malignant-Pro-008.jpg
WBC-Malignant-Pro-009.jpg
WBC-Malignant-Pro-010.jpg


In [ ]:
test_image_path = "/content/leukemia/Original/Pro/WBC-Malignant-Pro-001.jpg"

label, confidence = predict_image(test_image_path)
print("Prediction:", label)
print("Confidence:", f"{confidence:.2f}%")


Prediction: Pro
Confidence: 99.98%


Qwen-3-VL Integration

In [ ]:
!pip install -q transformers accelerate pillow


In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torch
from PIL import Image


In [ ]:
from huggingface_hub import login
login()


In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

model_id = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

vl_model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Qwen2-VL-2B loaded successfully")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen2-VL-2B loaded successfully


In [ ]:
from PIL import Image

def generate_patient_report(image_path, cnn_label, cnn_conf):
    image = Image.open(image_path).convert("RGB")

    # IMPORTANT:
    # 1. text must be STRING
    # 2. must contain <image> token
    # 3. images must be passed separately
    prompt = f"""<image>
Detected leukemia stage: {cnn_label}
Confidence: {cnn_conf:.2f}%

Explain this result in very simple, patient-friendly language.
Include:
- What this stage means
- Whether it suggests disease progression
- Whether doctor consultation is urgent
- General next steps

Do NOT provide diagnosis.
Do NOT mention AI or model names.
"""

    inputs = processor(
        text=prompt,
        images=[image],          # <-- MUST be a list
        return_tensors="pt"
    )

    inputs = {k: v.to(vl_model.device) for k, v in inputs.items()}

    outputs = vl_model.generate(
        **inputs,
        max_new_tokens=300
    )

    return processor.decode(outputs[0], skip_special_tokens=True)


LLaVA-1.5 for the explanation layer


In [ ]:
!pip install -q transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.6 MB/s eta 0:00:00


In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor
import torch

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)

vl_model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLaVA model loaded successfully")


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LLaVA model loaded successfully


In [ ]:
from PIL import Image

def generate_patient_report(image_path, cnn_label, cnn_conf):
    image = Image.open(image_path).convert("RGB")

    prompt = f"""
This is a microscopic blood smear image.

CNN result:
Stage: {cnn_label}
Confidence: {cnn_conf:.2f}%

Explain this result in very simple, patient-friendly language.
Mention:
- What this stage generally means
- Whether medical consultation is recommended
- General next steps

Do NOT give a diagnosis.
"""

    inputs = processor(
        images=image,          # ✅ image explicitly
        text=prompt,           # ✅ text explicitly
        return_tensors="pt"
    )

    inputs = {k: v.to(vl_model.device) for k, v in inputs.items()}

    outputs = vl_model.generate(
        **inputs,
        max_new_tokens=300
    )

    return processor.decode(outputs[0], skip_special_tokens=True)


In [ ]:
from PIL import Image

def generate_patient_report(image_path, cnn_label, cnn_conf):
    image = Image.open(image_path).convert("RGB")

    # 🔑 CRITICAL: <image> token MUST be present
    prompt = f"""<image>
This is a microscopic blood smear image.

CNN result:
Stage: {cnn_label}
Confidence: {cnn_conf:.2f}%

Explain this result in very simple, patient-friendly language.
Mention:
- What this stage generally means
- Whether medical consultation is recommended
- General next steps

Do NOT give a diagnosis.
"""

    inputs = processor(
        images=image,      # image explicitly
        text=prompt,       # text WITH <image>
        return_tensors="pt"
    )

    inputs = {k: v.to(vl_model.device) for k, v in inputs.items()}

    outputs = vl_model.generate(
        **inputs,
        max_new_tokens=300
    )

    return processor.decode(outputs[0], skip_special_tokens=True)


In [ ]:
test_image_path = "/content/leukemia/Original/Pro/WBC-Malignant-Pro-001.jpg"

label, confidence = predict_image(test_image_path)
report = generate_patient_report(test_image_path, label, confidence)

print("CNN Prediction:", label, f"({confidence:.2f}%)")
print("\n--- PATIENT FRIENDLY REPORT ---\n")
print(report)


CNN Prediction: Pro (99.98%)

--- PATIENT FRIENDLY REPORT ---


This is a microscopic blood smear image.

CNN result:
Stage: Pro
Confidence: 99.98%

Explain this result in very simple, patient-friendly language.
Mention:
- What this stage generally means
- Whether medical consultation is recommended
- General next steps

Do NOT give a diagnosis.

This is a microscopic blood smear image.


option step (difrent type of report)

In [72]:
from PIL import Image

def generate_patient_report(image_path, cnn_label, cnn_conf):
    image = Image.open(image_path).convert("RGB")

    prompt = f"""<image>
You are a medical assistant explaining a lab report to a patient.

The image is a microscopic blood smear.

Automated analysis result:
- Stage detected: {cnn_label}
- Confidence: {cnn_conf:.2f}%

Now write a clear, simple explanation for a patient:
1. What this stage generally means (in simple words)
2. Whether seeing a doctor soon is recommended
3. What the usual next steps are (tests or consultation)

Write in calm, non-alarming language.
Do NOT give a diagnosis.
Do NOT repeat the input text.
"""

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    )

    inputs = {k: v.to(vl_model.device) for k, v in inputs.items()}

    outputs = vl_model.generate(
        **inputs,
        max_new_tokens=150,      # shorter & clearer
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return processor.decode(outputs[0], skip_special_tokens=True)


PDF Medical Report Generation

Patient info (dummy)

CNN result

Confidence

Patient-friendly explanation

Date & disclaimer

In [75]:
!pip install -q reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.9 MB/s eta 0:00:00


In [76]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from datetime import datetime


In [77]:
def generate_pdf_report(
    output_path,
    patient_name,
    cnn_label,
    cnn_confidence,
    explanation_text
):
    doc = SimpleDocTemplate(output_path, pagesize=A4)
    styles = getSampleStyleSheet()
    elements = []

    # Title
    elements.append(Paragraph("<b>TruePatient.ai – Medical Screening Report</b>", styles["Title"]))
    elements.append(Spacer(1, 12))

    # Patient Info
    elements.append(Paragraph(f"<b>Patient Name:</b> {patient_name}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Date:</b> {datetime.now().strftime('%d %B %Y')}", styles["Normal"]))
    elements.append(Spacer(1, 12))

    # CNN Result
    elements.append(Paragraph("<b>Automated Image Analysis Result</b>", styles["Heading2"]))
    elements.append(Paragraph(f"Detected Stage: <b>{cnn_label}</b>", styles["Normal"]))
    elements.append(Paragraph(f"Confidence: <b>{cnn_confidence:.2f}%</b>", styles["Normal"]))
    elements.append(Spacer(1, 12))

    # Explanation
    elements.append(Paragraph("<b>Patient-Friendly Explanation</b>", styles["Heading2"]))
    elements.append(Paragraph(explanation_text.replace("\n", "<br/>"), styles["Normal"]))
    elements.append(Spacer(1, 20))

    # Disclaimer
    elements.append(Paragraph(
        "<b>Disclaimer:</b> This report is generated using an AI-based screening system. "
        "It is not a medical diagnosis. Please consult a qualified medical professional "
        "for confirmation and further guidance.",
        styles["Italic"]
    ))

    doc.build(elements)


In [78]:
pdf_path = "/content/TruePatient_Report.pdf"

generate_pdf_report(
    output_path=pdf_path,
    patient_name="Sample Patient",
    cnn_label=label,
    cnn_confidence=confidence,
    explanation_text=report
)

pdf_path


'/content/TruePatient_Report.pdf'